# Lab 1 — OAMP memory agent

Goal: create a memory-aware agent with `oracleagentmemory`, persist thread messages, retrieve a context card, and prove memory survives reconnect.


## 0. Connect to Oracle AI Database

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

from agentic_memory_workshop.connection import load_env, connect

load_env('../.env')
connection = connect()
print('Connected to Oracle AI Database')


## 1. Create the OAMP client

`OracleAgentMemory` owns the schema, embedder, and optional LLM-driven extraction. We start without automatic extraction so the primitives are visible.


In [ ]:
from oracleagentmemory.core import OracleAgentMemory
from agentic_memory_workshop.onnx import oamp_embedder

client = OracleAgentMemory(
    connection=connection,
    embedder=oamp_embedder(connection),
    extract_memories=False,
    schema_policy='create_if_necessary',
)
print('Memory client ready')


## 2. Register a user and an agent

OAMP scopes memory to users and agents. Re-running the cell is safe: existing records are ignored.


In [ ]:
USER_ID = 'workshop_user'
AGENT_ID = 'memory_agent'

for fn, entity_id, information in [
    (client.add_user, USER_ID, 'Workshop attendee learning agentic memory management.'),
    (client.add_agent, AGENT_ID, 'Assistant used in the Designing and Building Memory for AI Agents workshop.'),
]:
    try:
        fn(entity_id, information)
        print(f'Registered {entity_id}')
    except ValueError as exc:
        if 'already exists' in str(exc):
            print(f'Already exists: {entity_id}')
        else:
            raise


## 3. Add and search durable memory

This is the smallest useful memory loop: store a fact, embed it in Oracle, and retrieve it by meaning rather than exact text.


In [ ]:
memory_id = client.add_memory(
    'The workshop user prefers Python examples and wants database-backed agent memory.',
    user_id=USER_ID,
    agent_id=AGENT_ID,
)
print(f'Stored memory: {memory_id}')

results = client.search('what language should examples use?', user_id=USER_ID, max_results=5)
for result in results:
    print(f'[{result.record.record_type}] distance={result.distance:.3f}')
    print(' ', result.content)


## 4. Create a thread and read its context card

A thread is the unit you carry across turns. The context card is the compact XML-like block you paste into the next LLM call.


In [ ]:
from oracleagentmemory.apis.thread import Message

thread = client.create_thread(user_id=USER_ID, agent_id=AGENT_ID)
thread.add_messages([
    Message(role='user', content='I am building a restaurant concierge agent.'),
    Message(role='assistant', content='What does it need to remember?'),
    Message(role='user', content='Preferences, prior bookings, refund policies, and tool traces.'),
])
print(f'Thread ID: {thread.thread_id}')
print('Messages:', len(thread.get_messages()))
print('\nContext card:')
print(thread.get_context_card() or '(empty context card yet)')


## 5. Put the context card into every LLM call

The card only earns its keep if it reaches the model. `agent_turn` appends the user message to the thread, reads the current context card, sends it as the system prompt through LiteLLM, and stores the reply back into the same thread so the next turn sees it. The workshop helper `chat` fails over across the shared keys if one is rate limited.

Turn 2 only works because turn 1 is in memory: `Book the first one you suggested` means nothing outside the card.


In [ ]:
from agentic_memory_workshop.llm import chat, llm_status

llm_ready, llm_detail = llm_status()

SYSTEM_PROMPT = 'You are a restaurant concierge agent. Use the memory in the context card before you answer.'


def agent_turn(thread, user_text):
    """Send one turn with the thread context card as the system prompt and store the reply in the thread."""
    thread.add_messages([Message(role='user', content=user_text)])
    card = thread.get_context_card()
    reply = chat([
        {'role': 'system', 'content': f'{SYSTEM_PROMPT}\n\n{card.content or "(no memory yet)"}'},
        {'role': 'user', 'content': user_text},
    ])
    thread.add_messages([Message(role='assistant', content=reply)])
    return reply


print(f'LLM turn ready: {llm_detail}' if llm_ready else f'LLM turns will be skipped: {llm_detail}')


### Two turns, the second one dependent on the first

The replies come from the model, so the wording changes between runs. What stays fixed is the loop: read memory, call the model, write memory.


In [ ]:
if not llm_ready:
    print(f'Skipped the two-turn demo: {llm_detail}')
else:
    print('Turn 1:', agent_turn(thread, 'Suggest three restaurants for three people on Friday around 7pm. One of us avoids shellfish and we want a quiet table.'))
    print()
    print('Turn 2:', agent_turn(thread, 'Book the first one you suggested.'))
    print()
    print(f'Thread now holds {len(thread.get_messages())} messages')
    print('\nContext card after the exchange:')
    print(thread.get_context_card().content)


## 6. Optional: turn on extraction and summaries

With the LLM configured, this section creates a second client with `extract_memories=True`, enables context summaries, and lets OAMP extract facts from messages. `oamp_llm()` hands OAMP the same model and key failover as `chat`. The embedder still uses Oracle when `OAMP_EMBEDDER=oracle-db`.


In [ ]:
from agentic_memory_workshop.llm import oamp_llm

if not llm_ready:
    print(f'Skipped extraction: {llm_detail}')
else:
    smart_client = OracleAgentMemory(
        connection=connection,
        embedder=oamp_embedder(connection),
        llm=oamp_llm(),
        extract_memories=True,
        schema_policy='create_if_necessary',
    )
    smart_thread = smart_client.create_thread(
        user_id=USER_ID,
        agent_id=AGENT_ID,
        memory_extraction_frequency=2,
        memory_extraction_window=4,
        enable_context_summary=True,
        context_summary_update_frequency=2,
    )
    smart_thread.add_messages([
        Message(role='user', content='Remember that I prefer aisle tables and avoid shellfish.'),
        Message(role='assistant', content='Noted. I will use that before restaurant suggestions.'),
        Message(role='user', content='Also, I usually book for two people around 7pm.'),
        Message(role='assistant', content='I will include party size and timing in future context.'),
    ])
    print(smart_thread.get_context_card() or '(no card generated)')
    print('\nSummary:')
    print(smart_thread.get_summary())


## 7. Reconnect and recover the thread

The point of memory infrastructure is that a process restart does not erase the agent's operating context.


In [ ]:
saved_thread_id = thread.thread_id
connection.close()
new_connection = connect()
new_client = OracleAgentMemory(
    connection=new_connection,
    embedder=oamp_embedder(new_connection),
    extract_memories=False,
    schema_policy='create_if_necessary',
)
recovered = new_client.get_thread(saved_thread_id)
print(f'Recovered thread: {recovered.thread_id}')
print(f'Messages recovered: {len(recovered.get_messages())}')


## 8. Your turn: talk to the agent

Change the message in a `talk(...)` cell and run it again, as often as you like. Every run stores your message and the agent's reply in the thread, then prints the reply, the number of messages in the thread and the updated context card. Watch `<topics>`, `<summary>` and `<relevant_information>` (the facts and preferences OAMP extracts) change. Memory is per user, so facts from earlier sections (for example the shellfish preference) can show up.

In [ ]:
if not llm_ready:
    print(f'Skipped the chat section: {llm_detail}')
else:
    my_client = OracleAgentMemory(
        connection=new_connection,
        embedder=oamp_embedder(new_connection),
        llm=oamp_llm(),
        extract_memories=True,
        schema_policy='create_if_necessary',
    )
    my_thread = my_client.create_thread(
        user_id=USER_ID,
        agent_id=AGENT_ID,
        memory_extraction_frequency=2,
        memory_extraction_window=4,
        enable_context_summary=True,
        context_summary_update_frequency=2,
    )
    print(f'Thread: {my_thread.thread_id}')
    print(f'Messages in this thread: {len(my_thread.get_messages())}')
    card = my_thread.get_context_card()
    print('\nStarting context card:')
    print(card.content if card and card.content else '(empty card)')


def talk(user_text):
    if not llm_ready:
        print(f'Skipped the chat section: {llm_detail}')
        return
    print(f'You: {user_text}\n')
    reply = agent_turn(my_thread, user_text)
    print(f'Agent: {reply}\n')
    print(f'Messages in this thread: {len(my_thread.get_messages())}')
    print('\nContext card after this turn:')
    print(my_thread.get_context_card().content)


In [ ]:
talk('Hi! I am organizing a birthday dinner for six people next Saturday. Two of us are vegetarian.')

Now type your own messages: tell it a preference, change your mind, or ask what it remembers.

In [ ]:
talk('Actually we will be eight, and one guest avoids gluten.')

In [ ]:
talk('What do you remember about my dinner plans?')